# Assemble Transcripts

Loads all WhisperX word-level transcripts from `vad_new/{patient}/vad_data/*/transcription/whisperx/audio.json`
and assembles them into a single DataFrame saved to `vad_new/{patient}/{patient}_transcripts.csv`.

**One row per word.** Each row carries:
- word text, timing (seconds from audio start, UTC, Blackrock sample index)
- segment context (segment text, speaker, segment bounds)
- interval identifiers (interval_id, toc, patient) for indexing back into neural/audio/video data
- rolling context string of the preceding N words within the interval (for GPT-2 forward passes)

In [1]:
import json
from collections import deque
from pathlib import Path

import pandas as pd

In [16]:
# ── Config ─────────────────────────────────────────────────────────────────────
PATIENT = "YFA"
VAD_ROOT = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{PATIENT}")
VAD_DATA_DIR = VAD_ROOT / "vad_data"

# Context window: number of preceding words kept as context string.
# 200 words ≈ 250–300 GPT-2 tokens on typical English text.
# Context resets at interval boundaries.
CONTEXT_WORDS = 200

# Blackrock recording sample rate (used to convert seconds → BR sample index)
BR_FS = 30_000

OUT_PATH = VAD_ROOT / f"{PATIENT}_transcripts.csv"
print("output:", OUT_PATH)

output: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/YFA_transcripts.csv


In [17]:
def load_interval_words(interval_dir: Path, patient: str) -> list[dict]:
    """Return a list of word-level dicts for one interval, or [] on any failure."""
    meta_path = interval_dir / "metadata.json"
    tx_json = interval_dir / "transcription" / "whisperx" / "audio.json"
    success = interval_dir / "transcription" / "_SUCCESS"

    if not (success.exists() and meta_path.exists() and tx_json.exists()):
        return []

    with open(meta_path) as f:
        meta = json.load(f)
    with open(tx_json) as f:
        wdata = json.load(f)

    interval_id = meta["interval_id"]
    toc = meta["toc"]
    utc_start = pd.Timestamp(meta["utc_ts"])
    utc_end = pd.Timestamp(meta["utc_te"])
    br_ts = int(meta["br_ts"])
    br_te = int(meta["br_te"])
    interval_dur_s = float(meta["duration_s"])

    # Build a word → segment mapping from the segments list
    segments = wdata.get("segments", [])
    word_to_seg: dict[int, tuple] = {}  # word_position → (seg_idx, seg)
    global_word_pos = 0
    for seg_idx, seg in enumerate(segments):
        for _ in seg.get("words", []):
            word_to_seg[global_word_pos] = (seg_idx, seg)
            global_word_pos += 1

    words = wdata.get("word_segments", [])
    rows = []
    context_window: deque[str] = deque(maxlen=CONTEXT_WORDS)

    for wi, w in enumerate(words):
        word_text = str(w.get("word", "")).strip()
        word_start_s = float(w.get("start", 0.0))
        word_end_s = float(w.get("end", word_start_s))
        word_score = float(w.get("score", float("nan")))

        # Segment membership
        seg_idx, seg = word_to_seg.get(wi, (None, {}))
        segment_text = str(seg.get("text", "")).strip()
        segment_start_s = float(seg.get("start", word_start_s))
        segment_end_s = float(seg.get("end", word_end_s))
        speaker = str(seg.get("speaker", ""))

        # Absolute timestamps
        utc_word_start = utc_start + pd.Timedelta(seconds=word_start_s)
        utc_word_end = utc_start + pd.Timedelta(seconds=word_end_s)
        br_word_start = br_ts + int(round(word_start_s * BR_FS))
        br_word_end = br_ts + int(round(word_end_s * BR_FS))

        # Context is the words seen so far in this interval (before appending current word)
        context = " ".join(context_window)
        context_window.append(word_text)

        rows.append({
            # Identity
            "patient": patient,
            "interval_id": interval_id,
            "toc": toc,
            # Word
            "word": word_text,
            "word_start_s": word_start_s,
            "word_end_s": word_end_s,
            "word_score": word_score,
            # Segment
            "segment_idx": seg_idx,
            "segment_text": segment_text,
            "segment_start_s": segment_start_s,
            "segment_end_s": segment_end_s,
            "speaker": speaker,
            # Interval timing
            "interval_utc_start": utc_start,
            "interval_utc_end": utc_end,
            "interval_dur_s": interval_dur_s,
            "br_ts": br_ts,
            "br_te": br_te,
            # Absolute word timing
            "utc_word_start": utc_word_start,
            "utc_word_end": utc_word_end,
            "br_word_start": br_word_start,
            "br_word_end": br_word_end,
            # Context (preceding words within this interval)
            "context": context,
        })

    return rows

In [18]:
# ── Scan and assemble ──────────────────────────────────────────────────────────
all_rows = []
n_intervals_ok = 0
n_intervals_skipped = 0

interval_dirs = sorted(VAD_DATA_DIR.iterdir())
for idir in interval_dirs:
    if not idir.is_dir():
        continue
    rows = load_interval_words(idir, PATIENT)
    if rows:
        all_rows.extend(rows)
        n_intervals_ok += 1
    else:
        n_intervals_skipped += 1

df = pd.DataFrame(all_rows)

print(f"intervals loaded:  {n_intervals_ok}")
print(f"intervals skipped: {n_intervals_skipped}  (no _SUCCESS or missing files)")
print(f"total words:       {len(df)}")
if not df.empty:
    print(f"tocs covered:      {df.toc.nunique()}")
    print(f"date range:        {df.interval_utc_start.min():%Y-%m-%d}  →  {df.interval_utc_end.max():%Y-%m-%d}")

intervals loaded:  1030
intervals skipped: 53  (no _SUCCESS or missing files)
total words:       610928
tocs covered:      11
date range:        2024-04-23  →  2024-05-01


In [19]:
# ── Quick look ─────────────────────────────────────────────────────────────────
df.head(10)

,patient,interval_id,toc,word,word_start_s,word_end_s,word_score,segment_idx,segment_text,segment_start_s,...,interval_utc_start,interval_utc_end,interval_dur_s,br_ts,br_te,utc_word_start,utc_word_end,br_word_start,br_word_end,context
0,YFA,20240423-124841_0004,20240423-124841,you,43.904,44.675,0.607,0,you,43.904,...,2024-04-23 20:24:41.962,2024-04-23 20:33:47.069,545.107,2175209868,2191560303,2024-04-23 20:25:25.866,2024-04-23 20:25:26.637,2176526988,2176550118,
1,YFA,20240423-171352_0000,20240423-171352,Leave,0.131,0.271,0.223,0,Leave everything behind that you don't need!,0.131,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.205,2024-04-23 22:13:53.345,2371743269,2371747469,
2,YFA,20240423-171352_0000,20240423-171352,everything,0.311,0.571,0.131,0,Leave everything behind that you don't need!,0.131,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.385,2024-04-23 22:13:53.645,2371748669,2371756469,Leave
3,YFA,20240423-171352_0000,20240423-171352,behind,0.591,0.771,0.239,0,Leave everything behind that you don't need!,0.131,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.665,2024-04-23 22:13:53.845,2371757069,2371762469,Leave everything
4,YFA,20240423-171352_0000,20240423-171352,that,0.791,0.891,0.708,0,Leave everything behind that you don't need!,0.131,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.865,2024-04-23 22:13:53.965,2371763069,2371766069,Leave everything behind
5,YFA,20240423-171352_0000,20240423-171352,you,0.911,1.012,0.788,0,Leave everything behind that you don't need!,0.131,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.985,2024-04-23 22:13:54.086,2371766669,2371769699,Leave everything behind that
6,YFA,20240423-171352_0000,20240423-171352,don't,1.032,1.292,0.555,0,Leave everything behind that you don't need!,0.131,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:54.106,2024-04-23 22:13:54.366,2371770299,2371778099,Leave everything behind that you
7,YFA,20240423-171352_0000,20240423-171352,need!,1.352,1.912,0.631,0,Leave everything behind that you don't need!,0.131,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:54.426,2024-04-23 22:13:54.986,2371779899,2371796699,Leave everything behind that you don't
8,YFA,20240423-171352_0000,20240423-171352,Oh,2.032,2.272,0.741,1,"Oh no, Hinata!",2.032,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:55.106,2024-04-23 22:13:55.346,2371800299,2371807499,Leave everything behind that you don't need!
9,YFA,20240423-171352_0000,20240423-171352,"no,",2.312,2.512,0.767,1,"Oh no, Hinata!",2.032,...,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:55.386,2024-04-23 22:13:55.586,2371808699,2371814699,Leave everything behind that you don't need! Oh


In [20]:
# ── Per-interval word count summary ───────────────────────────────────────────
summary = (
    df.groupby("interval_id")
    .agg(
        n_words=("word", "count"),
        n_segments=("segment_idx", "nunique"),
        avg_score=("word_score", "mean"),
        dur_s=("interval_dur_s", "first"),
        toc=("toc", "first"),
    )
    .reset_index()
    .sort_values("interval_id")
)
print(summary.describe())
summary.head(20)

           n_words   n_segments    avg_score        dur_s
count  1030.000000  1030.000000  1030.000000  1030.000000
mean    593.133981    78.158252     0.494352   308.952083
std    1055.359077   133.662758     0.114492   493.414056
min       1.000000     1.000000     0.064000     0.300000
25%      16.000000     2.000000     0.435859    15.625000
50%     129.000000    19.000000     0.482401    90.750000
75%     562.750000    80.750000     0.547986   326.314250
max    5308.000000   679.000000     0.980000  1799.955000


,interval_id,n_words,n_segments,avg_score,dur_s,toc
0,20240423-124841_0004,1,1,0.607000,545.107,20240423-124841
1,20240423-171352_0000,1350,158,0.627564,599.600,20240423-171352
2,20240423-171352_0001,2351,329,0.611782,1599.834,20240423-171352
3,20240423-171352_0002,161,23,0.619578,122.100,20240423-171352
4,20240423-171352_0003,1371,248,0.570530,839.015,20240423-171352
5,20240423-171352_0005,5,2,0.488800,11.500,20240423-171352
6,20240423-171352_0006,1016,162,0.444336,831.666,20240423-171352
7,20240423-171352_0007,412,56,0.552231,259.856,20240423-171352
8,20240423-171352_0008,11,2,0.305545,3.300,20240423-171352
9,20240423-171352_0009,273,49,0.577293,227.609,20240423-171352


In [21]:
# ── Save ───────────────────────────────────────────────────────────────────────
df.to_csv(OUT_PATH, index=False)
print(f"saved {len(df)} rows to {OUT_PATH}")
print(f"columns: {list(df.columns)}")

saved 610928 rows to /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/YFA_transcripts.csv
columns: ['patient', 'interval_id', 'toc', 'word', 'word_start_s', 'word_end_s', 'word_score', 'segment_idx', 'segment_text', 'segment_start_s', 'segment_end_s', 'speaker', 'interval_utc_start', 'interval_utc_end', 'interval_dur_s', 'br_ts', 'br_te', 'utc_word_start', 'utc_word_end', 'br_word_start', 'br_word_end', 'context']
